<a href="https://colab.research.google.com/github/Andrew-Lever/Programming-BigDataAnalytics/blob/main/Data_Joining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

# 1. Load the core datasets
df_loan_applications = pd.read_csv('loan_applications.csv')[['SK_ID_CURR', 'TARGET','AMT_INCOME_TOTAL', 'AMT_CREDIT','AMT_ANNUITY', 'AMT_GOODS_PRICE','DAYS_EMPLOYED','DAYS_REGISTRATION','DAYS_ID_PUBLISH','OWN_CAR','OWN_HOME']]
df_previous_loan = pd.read_csv('previous_loan_applications.csv')[['SK_ID_CURR', 'SK_ID_PREV', 'AMT_ANNUITY','AMT_APPLICATION','AMT_CREDIT','NAME_CONTRACT_STATUS']]
df_instalment_payments = pd.read_csv('instalment_payments.csv')

# 2. Immediately drop the granular columns causing duplication from instalments
columns_to_remove = ['DAYS_ENTRY_PAYMENT', 'DAYS_INSTALMENT', 'NUM_INSTALMENT_NUMBER']
df_instalment_clean = df_instalment_payments.drop(columns=[col for col in columns_to_remove if col in df_instalment_payments.columns])

In [ ]:
# 3. Merge applications with previous loans
joined_df = pd.merge(df_loan_applications, df_previous_loan, on="SK_ID_CURR", how="inner")

# 4. Merge with cleaned instalment payments and drop duplicates to stay at the loan application grain
joined_df = pd.merge(joined_df, df_instalment_clean, on=["SK_ID_CURR", "SK_ID_PREV"], how="left")
joined_df = joined_df.drop_duplicates(subset=['SK_ID_CURR', 'SK_ID_PREV'])

print(f"Corrected non-duplicated merged dataset shape: {joined_df.shape}")
display(joined_df.head())

Corrected non-duplicated merged dataset shape: (124037, 19)


,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT_x,AMT_ANNUITY_x,AMT_GOODS_PRICE,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR,OWN_HOME,SK_ID_PREV,AMT_ANNUITY_y,AMT_APPLICATION,AMT_CREDIT_y,NAME_CONTRACT_STATUS,NUM_INSTALMENT_VERSION,AMT_INSTALMENT,AMT_PAYMENT
0,100008,0,99000.0,490495.5,27517.5,454500.0,-1588,-4970,-477,1,1,2218188,17885.835,162598.5,162598.5,Approved,1.0,17885.835,17885.835
10,100008,0,99000.0,490495.5,27517.5,454500.0,-1588,-4970,-477,1,1,1757431,NaN,0.0,0.0,Canceled,NaN,NaN,NaN
11,100008,0,99000.0,490495.5,27517.5,454500.0,-1588,-4970,-477,1,1,1186888,8019.090,44455.5,39955.5,Approved,1.0,8019.090,8019.090
17,100008,0,99000.0,490495.5,27517.5,454500.0,-1588,-4970,-477,1,1,2613879,25309.575,450000.0,501975.0,Approved,1.0,25309.575,25309.575
25,100008,0,99000.0,490495.5,27517.5,454500.0,-1588,-4970,-477,1,1,1907290,12144.285,121455.0,109309.5,Approved,1.0,12144.285,12144.285


### Understanding Client vs. Previous Loan Grain

One client (`SK_ID_CURR`) can have multiple previous loans (`SK_ID_PREV`). Because of this, we currently have **124,037 rows** (previous loan level) instead of **25,526 rows** (unique clients).

To build a model that predicts defaults on the *current* application level, we should collapse these previous loans by grouping by `SK_ID_CURR` and calculating summary statistics (e.g., number of previous loans, average amount credited, or approval rates).

In [ ]:
# Grouping previous loans to collapse the dataset down to exactly one row per unique applicant
# We will aggregate numeric details by their mean/sum, and count contract statuses.

# 1. Create status flags for previous applications
joined_df['PREV_APPROVED'] = (joined_df['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
joined_df['PREV_REFUSED'] = (joined_df['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)

# 2. Define the aggregation rules
agg_rules = {
    'TARGET': 'first',             # Target remains the same for the client
    'AMT_INCOME_TOTAL': 'first',   # Application features remain the same
    'AMT_CREDIT_x': 'first',
    'AMT_ANNUITY_x': 'first',
    'AMT_GOODS_PRICE': 'first',
    'DAYS_EMPLOYED': 'first',
    'DAYS_REGISTRATION': 'first',
    'DAYS_ID_PUBLISH': 'first',
    'OWN_CAR': 'first',
    'OWN_HOME': 'first',

    # Aggregate historical previous loan metrics per client
    'SK_ID_PREV': 'count',         # Total number of previous loan applications
    'AMT_APPLICATION': 'mean',     # Average amount requested in the past
    'AMT_CREDIT_y': 'mean',        # Average credit amount granted in the past
    'AMT_PAYMENT': 'sum',          # Total historical installments paid
    'PREV_APPROVED': 'sum',        # Total approved loans
    'PREV_REFUSED': 'sum'          # Total refused loans
}

# Rename columns during aggregation for clarity
client_level_df = joined_df.groupby('SK_ID_CURR').agg(agg_rules).reset_index()
client_level_df.rename(columns={
    'SK_ID_PREV': 'PREV_APP_COUNT',
    'AMT_APPLICATION': 'PREV_AVG_APP_AMT',
    'AMT_CREDIT_y': 'PREV_AVG_CREDIT_AMT',
    'AMT_PAYMENT': 'PREV_TOTAL_PAYMENTS_MADE'
}, inplace=True)

print(f"Final client-level dataset shape: {client_level_df.shape}")
print(f"Number of unique clients: {client_level_df['SK_ID_CURR'].nunique()}")

# Verify client 100008 is now represented by exactly one row
display(client_level_df[client_level_df['SK_ID_CURR'] == 100008])

Final client-level dataset shape: (25526, 17)
Number of unique clients: 25526


,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT_x,AMT_ANNUITY_x,AMT_GOODS_PRICE,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR,OWN_HOME,PREV_APP_COUNT,PREV_AVG_APP_AMT,PREV_AVG_CREDIT_AMT,PREV_TOTAL_PAYMENTS_MADE,PREV_APPROVED,PREV_REFUSED
0,100008,0,99000.0,490495.5,27517.5,454500.0,-1588,-4970,-477,1,1,5,155701.8,162767.7,63358.785,4,0


In [ ]:
# 1. Compute granular metrics at the transaction level before aggregating
# This allows us to capture payment behavior without duplicating rows.
df_instalment_payments['PAYMENT_DELAY'] = df_instalment_payments['DAYS_ENTRY_PAYMENT'] - df_instalment_payments['DAYS_INSTALMENT']
df_instalment_payments['PAYMENT_UNDER_PAID'] = df_instalment_payments['AMT_INSTALMENT'] - df_instalment_payments['AMT_PAYMENT']

# 2. Aggregate these installment behaviors per (SK_ID_CURR, SK_ID_PREV)
df_inst_grouped = df_instalment_payments.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
    MAX_INSTALLMENTS=('NUM_INSTALMENT_NUMBER', 'max'),
    AVG_PAYMENT_DELAY=('PAYMENT_DELAY', 'mean'),
    NUM_LATE_PAYMENTS=('PAYMENT_DELAY', lambda x: (x > 0).sum()),
    TOTAL_UNDERPAYMENT=('PAYMENT_UNDER_PAID', 'sum')
).reset_index()

# 3. Join these aggregated previous-loan metrics back to our joined_df
joined_df_with_inst = pd.merge(df_loan_applications, df_previous_loan, on="SK_ID_CURR", how="inner")
joined_df_with_inst = pd.merge(joined_df_with_inst, df_inst_grouped, on=["SK_ID_CURR", "SK_ID_PREV"], how="left")

# 4. Collapse everything to the unique client grain (one row per SK_ID_CURR)
# Now we define our new aggregation rules including the newly generated variables!
joined_df_with_inst['PREV_APPROVED'] = (joined_df_with_inst['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
joined_df_with_inst['PREV_REFUSED'] = (joined_df_with_inst['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)

agg_rules_updated = {
    'TARGET': 'first',
    'AMT_INCOME_TOTAL': 'first',
    'AMT_CREDIT_x': 'first',
    'AMT_ANNUITY_x': 'first',
    'AMT_GOODS_PRICE': 'first',
    'DAYS_EMPLOYED': 'first',
    'DAYS_REGISTRATION': 'first',
    'DAYS_ID_PUBLISH': 'first',
    'OWN_CAR': 'first',
    'OWN_HOME': 'first',

    # Aggregating the historical installment features
    'SK_ID_PREV': 'count',
    'AMT_APPLICATION': 'mean',
    'AMT_CREDIT_y': 'mean',
    'MAX_INSTALLMENTS': 'mean',        # Average number of installments across previous loans
    'AVG_PAYMENT_DELAY': 'mean',      # Average payment delay across all previous loans
    'NUM_LATE_PAYMENTS': 'sum',       # Total number of late payments across history
    'TOTAL_UNDERPAYMENT': 'sum',      # Total underpayment amount across history
    'PREV_APPROVED': 'sum',
    'PREV_REFUSED': 'sum'
}

client_level_df_rich = joined_df_with_inst.groupby('SK_ID_CURR').agg(agg_rules_updated).reset_index()
client_level_df_rich.rename(columns={
    'SK_ID_PREV': 'PREV_APP_COUNT',
    'AMT_APPLICATION': 'PREV_AVG_APP_AMT',
    'AMT_CREDIT_y': 'PREV_AVG_CREDIT_AMT'
}, inplace=True)

# Impute missing values for clients with no previous installment/payment history
X_cols_rich = [col for col in client_level_df_rich.columns if col not in ['SK_ID_CURR', 'TARGET']]
client_level_df_rich[X_cols_rich] = client_level_df_rich[X_cols_rich].fillna(0)

# Save the final richer dataset
client_level_df_rich.to_csv('Final_Cleaned_Client_Dataset.csv', index=False)

print(f"Success! Your enriched dataset with historical behavior has been generated.")
print(f"Final Dimensions: {client_level_df_rich.shape[0]} rows, {client_level_df_rich.shape[1]} columns.")
display(client_level_df_rich.head())

Success! Your enriched dataset with historical behavior has been generated.
Final Dimensions: 25526 rows, 20 columns.


,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT_x,AMT_ANNUITY_x,AMT_GOODS_PRICE,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR,OWN_HOME,PREV_APP_COUNT,PREV_AVG_APP_AMT,PREV_AVG_CREDIT_AMT,MAX_INSTALLMENTS,AVG_PAYMENT_DELAY,NUM_LATE_PAYMENTS,TOTAL_UNDERPAYMENT,PREV_APPROVED,PREV_REFUSED
0,100008,0,99000.0,490495.5,27517.5,454500.0,-1588,-4970,-477,1,1,5,155701.800000,162767.700000,8.500000,17.070076,1.0,11986.155,4,0
1,100048,0,202500.0,604152.0,29196.0,540000.0,-475,-3148,-513,1,1,4,394875.000000,432694.125000,22.666667,-10.183333,1.0,0.000,3,0
2,100054,0,99000.0,260640.0,26838.0,225000.0,-5332,-9197,-3283,1,1,5,85999.500000,81145.800000,11.250000,-7.329365,3.0,13165.740,4,1
3,100080,0,157500.0,450000.0,22500.0,450000.0,-3154,-8118,-4143,0,0,7,155487.857143,175481.357143,15.750000,-1.615530,6.0,0.000,4,0
4,100100,0,202500.0,796396.5,38443.5,643500.0,-1094,-3134,-4180,1,1,5,137855.700000,189446.400000,34.600000,-10.601238,18.0,70315.650,5,0


In [ ]:
import pandas as pd

# Ensure we use the rich client-level dataset that includes our payment behavior aggregations
final_client_df = client_level_df_rich.copy()

# Fill any remaining NaNs in the aggregated columns with 0 (e.g. clients who had no previous loans)
X_cols = [col for col in final_client_df.columns if col not in ['SK_ID_CURR', 'TARGET']]
final_client_df[X_cols] = final_client_df[X_cols].fillna(0)

# Save our final single-observation-per-client dataset with all rich metrics to a CSV file
final_output_path = 'Final_Cleaned_Client_Dataset.csv'
final_client_df.to_csv(final_output_path, index=False)

print(f"Success! Your final clean, rich client-level dataset is exported to: {final_output_path}")
print(f"Final Dimensions: {final_client_df.shape[0]} rows (one per unique client), {final_client_df.shape[1]} columns.")
display(final_client_df.head())


Success! Your final clean, rich client-level dataset is exported to: Final_Cleaned_Client_Dataset.csv
Final Dimensions: 25526 rows (one per unique client), 20 columns.


,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT_x,AMT_ANNUITY_x,AMT_GOODS_PRICE,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR,OWN_HOME,PREV_APP_COUNT,PREV_AVG_APP_AMT,PREV_AVG_CREDIT_AMT,MAX_INSTALLMENTS,AVG_PAYMENT_DELAY,NUM_LATE_PAYMENTS,TOTAL_UNDERPAYMENT,PREV_APPROVED,PREV_REFUSED
0,100008,0,99000.0,490495.5,27517.5,454500.0,-1588,-4970,-477,1,1,5,155701.800000,162767.700000,8.500000,17.070076,1.0,11986.155,4,0
1,100048,0,202500.0,604152.0,29196.0,540000.0,-475,-3148,-513,1,1,4,394875.000000,432694.125000,22.666667,-10.183333,1.0,0.000,3,0
2,100054,0,99000.0,260640.0,26838.0,225000.0,-5332,-9197,-3283,1,1,5,85999.500000,81145.800000,11.250000,-7.329365,3.0,13165.740,4,1
3,100080,0,157500.0,450000.0,22500.0,450000.0,-3154,-8118,-4143,0,0,7,155487.857143,175481.357143,15.750000,-1.615530,6.0,0.000,4,0
4,100100,0,202500.0,796396.5,38443.5,643500.0,-1094,-3134,-4180,1,1,5,137855.700000,189446.400000,34.600000,-10.601238,18.0,70315.650,5,0


In [ ]:
final_client_df.to_csv('Final_Clean_Loan_Data.csv')

In [ ]:
print(final_client_df.head().to_string())

    SK_ID_CURR  TARGET  AMT_INCOME_TOTAL  AMT_CREDIT_x  AMT_ANNUITY_x  AMT_GOODS_PRICE  DAYS_EMPLOYED  DAYS_REGISTRATION  DAYS_ID_PUBLISH  OWN_CAR  OWN_HOME  SK_ID_PREV  AMT_ANNUITY_y  AMT_APPLICATION  AMT_CREDIT_y NAME_CONTRACT_STATUS  NUM_INSTALMENT_VERSION  AMT_INSTALMENT  AMT_PAYMENT
0       100008       0           99000.0      490495.5        27517.5         454500.0          -1588              -4970             -477        1         1     2218188      17885.835         162598.5      162598.5             Approved                     1.0       17885.835    17885.835
10      100008       0           99000.0      490495.5        27517.5         454500.0          -1588              -4970             -477        1         1     1757431            NaN              0.0           0.0             Canceled                     NaN             NaN          NaN
11      100008       0           99000.0      490495.5        27517.5         454500.0          -1588              -4970             

In [ ]:
print(final_client_df.tail().to_string())

        SK_ID_CURR_x  TARGET  AMT_INCOME_TOTAL  AMT_CREDIT_x  AMT_ANNUITY_x  AMT_GOODS_PRICE  DAYS_EMPLOYED  DAYS_REGISTRATION  DAYS_ID_PUBLISH  OWN_CAR  OWN_HOME  SK_ID_PREV  AMT_ANNUITY_y  AMT_APPLICATION  AMT_CREDIT_y NAME_CONTRACT_STATUS  SK_ID_CURR_y  NUM_INSTALMENT_VERSION  NUM_INSTALMENT_NUMBER  DAYS_INSTALMENT  DAYS_ENTRY_PAYMENT  AMT_INSTALMENT  AMT_PAYMENT
931968        456255       0          157500.0      675000.0        49117.5         675000.0          -1262              -5128             -410        0         0     2631384      54022.140        1170000.0     1271929.5             Approved        456255                     3.0                     12           -426.0              -435.0       27489.690    27489.690
931969        456255       0          157500.0      675000.0        49117.5         675000.0          -1262              -5128             -410        0         0     2631384      54022.140        1170000.0     1271929.5             Approved        456255       